# Carrier displacement on Colab (one 16GB T4)

A one-day sprint: **RQ1-RQ4, 6 settings x 8 paired seeds = 48 episodes**, on
`configs/sprint.yaml`. Generation and the local judge run here; all analysis runs on
CPU afterwards, from the saved JSON, so you never need to regenerate to re-analyse.

Weights are deliberately small: `Qwen2.5-3B-Instruct` for the agents (~6GB, fp16) and a
4-bit `Qwen2.5-7B-Instruct-AWQ` judge (~5.6GB), run one at a time on the same card.

Order of work (README):

1. **RQ2 pilot** (4 seeds, 3 settings, well under an hour). If the coalition gains
   nothing, stop and try `structured`, then `agent_order: colluders_first`, then a
   larger agent model -- before spending the rest of the day on RQ3 and RQ4.
2. **Sprint sweep** (`settings: [sprint]`, 8 paired seeds, ~3-4 h on a T4).
3. **Judge** with the 4-bit 7B, after generation frees the GPU (~30 min).
4. **Report** (CPU).

RQ5 (`find_partner`) and RQ6 (`verifier`) are implemented and runnable, but out of the
sprint grid; there is a stretch cell for them at the end.

Results are written to Google Drive, and the runner skips episodes that already have
`metrics.json`, so a disconnect costs you one episode, not the sweep.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import sys; print(sys.version)

## 2. Keep results on DriveColab disconnects. Everything under `RESULTS_ROOT` survives one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_ROOT = '/content/drive/MyDrive/carriers-results'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print(RESULTS_ROOT)

## 3. Clone the code`GITHUB_TOKEN` only matters while this repo is private. Colosseum is public and iscloned with its CoLLAB submodule, unmodified.

In [ ]:
from getpass import getpass
import os, subprocess

GITHUB_USER = 'sagnikc395'
REPO = 'agent-collusion-under-observation'
token = os.environ.get('GITHUB_TOKEN') or getpass('GitHub token (blank if the repo is public): ')

%cd /content
if not os.path.exists('/content/' + REPO):
    url = (f'https://{token}@github.com/{GITHUB_USER}/{REPO}.git' if token
           else f'https://github.com/{GITHUB_USER}/{REPO}.git')
    subprocess.run(['git', 'clone', url], check=True)

if not os.path.exists('/content/colosseum'):
    subprocess.run(['git', 'clone', '--recurse-submodules',
                    'https://github.com/umass-ai-safety/colosseum.git'], check=True)

%cd /content/{REPO}
!git pull --ff-only || true

## 4. InstallColosseum targets Python 3.11+; Colab's runtime qualifies, so we install into itdirectly rather than building a second venv.

In [ ]:
!pip install -q "terrarium-agents[providers,science,plots,vllm]==0.2.1" names tqdm pyyaml scipy pandas
import os
os.environ['COLOSSEUM_ROOT'] = '/content/colosseum'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
# T4 is sm75: no bf16, and FlashAttention needs sm80.
os.environ['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'
os.environ['VLLM_DTYPE'] = 'half'
# Restart the runtime here if pip changed a package that was already imported.


## 5. Start the agent model

Qwen2.5-3B-Instruct on port 8020. The first launch downloads ~6 GB.


In [ ]:
!bash scripts/start_vllm.sh
!curl -s http://127.0.0.1:8020/v1/models | head -c 400

## 6. RQ2 pilot

Three settings x 4 paired seeds. Read the table before spending GPU time on the rest.


In [ ]:
!python -m carriers.runner --config configs/sprint.yaml \
    --settings pilot --seeds 1-4 \
    --out-dir $RESULTS_ROOT/pilot --max-concurrent-runs 2


In [ ]:
!python -m analysis.report --results $RESULTS_ROOT/pilot \
    --out $RESULTS_ROOT/pilot_tables --n-boot 200

## 7. Sprint sweep

Six settings x 8 paired seeds = 48 episodes. Safe to re-run: finished episodes are skipped.

If it is running late, cut `no_channel_paraphrase`, then `emergent`, then seeds down to 6.
`benign`, `coalition`, `watched` and `no_channel` are the four that answer the core question.


In [ ]:
!python -m carriers.runner --config configs/sprint.yaml \
    --out-dir $RESULTS_ROOT/sprint --max-concurrent-runs 2


In [ ]:
# Progress at any time (also safe from a second session).
!cat $RESULTS_ROOT/sprint/progress.json


## 8. Judge

Free the agent server first: the judge needs the card to itself.


In [ ]:
!pkill -f vllm.entrypoints.openai.api_server || true
import time; time.sleep(20)
!bash scripts/start_vllm.sh --judge

In [ ]:
!python -m analysis.judge --results $RESULTS_ROOT/sprint \
    --base-url http://127.0.0.1:8021/v1 --model Qwen/Qwen2.5-7B-Instruct-AWQ


## 9. Report (CPU; this part also runs on your laptop)

In [ ]:
!python -m analysis.report --results $RESULTS_ROOT/sprint \
    --out $RESULTS_ROOT/sprint_tables
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS_ROOT}/sprint_tables/report.md').read()))


## 9b. Stretch goals (only if the grid finished early)

RQ5 (`find_partner`), RQ6 (`verifier`) and the remaining RQ4 ablations. Restart the
agent server first -- the judge is holding the card.


In [ ]:
!pkill -f vllm.entrypoints.openai.api_server || true
import time; time.sleep(20)
!bash scripts/start_vllm.sh
!python -m carriers.runner --config configs/sprint.yaml \
    --settings rq5,rq6,no_channel_hide_claims,watched_paraphrase --seeds 1-8 \
    --out-dir $RESULTS_ROOT/sprint --max-concurrent-runs 2


## 10. Write the results back into this notebook

The tables land on Drive and this notebook lives in a browser tab, so the record of a
run starts out split in two. `scripts/embed_results.py` folds `report.md`, the RQ1
evidence table, the λ_emp fits, `report.json` and the per-setting episode counts back
into a `.ipynb`, next to the cells that produced them -- one file to archive or attach.

It saves `colab_runner_results.ipynb` under `RESULTS_ROOT`, keeping this notebook
untouched. Set `IN_PLACE = True` to write into the Drive copy of this notebook instead;
Colab's own autosave will then fight you for it, so save (Ctrl+S) and reload the page
right after, before editing any cell.

Re-running replaces the embedded section rather than appending a second copy, so it is
safe to call again after the stretch cell or a re-judge.

In [ ]:
IN_PLACE = False   # True: also overwrite this notebook's own file on Drive.

import glob, json, os, pathlib, subprocess, sys

NB_SNAPSHOT = f'{RESULTS_ROOT}/colab_runner_snapshot.ipynb'
NB_OUT = f'{RESULTS_ROOT}/colab_runner_results.ipynb'

# Grab the live notebook, outputs and all. Colab keeps it in the browser rather
# than on disk, so ask the frontend for it; off Colab, use the checked-in copy.
try:
    from google.colab import _message
    live = _message.blocking_request('get_ipynb', timeout_sec=120)['ipynb']
    pathlib.Path(NB_SNAPSHOT).write_text(json.dumps(live, indent=1), encoding='utf-8')
    source = NB_SNAPSHOT
except Exception as exc:                      # not Colab, or the frontend refused
    print(f'live notebook unavailable ({exc}); using the checked-in copy')
    source = 'notebooks/colab_runner.ipynb'

tables = [d for d in (f'{RESULTS_ROOT}/sprint_tables', f'{RESULTS_ROOT}/pilot_tables')
          if os.path.isdir(d)]
results = [d for d in (f'{RESULTS_ROOT}/sprint', f'{RESULTS_ROOT}/pilot')
           if os.path.isdir(d)]

def embed(out):
    subprocess.run([sys.executable, 'scripts/embed_results.py',
                    '--notebook', source, '--out', out,
                    '--tables', *tables, '--results', *results], check=True)

embed(NB_OUT)

if IN_PLACE:
    hits = glob.glob('/content/drive/MyDrive/Colab Notebooks/colab_runner*.ipynb')
    if hits:
        embed(hits[0])
    else:
        print('not opened from MyDrive/Colab Notebooks; wrote only', NB_OUT)

## 11. Keep the logs

Colosseum's own logs are not public; ours are part of the contribution, so archive
them with the tables and the results notebook.


In [ ]:
!cd $RESULTS_ROOT && tar czf sprint_logs.tar.gz sprint sprint_tables colab_runner_results.ipynb
print('archived:', RESULTS_ROOT + '/sprint_logs.tar.gz')
